In [9]:
import ollama
from tools import TOOLS

In [ ]:
def decide_tool_llm(query: str):
    prompt = f"""
You are a strict tool selector. Select the most appropriate tool for the given query from the list of allowed tools.
The given query can be very loose and may not directly mention the tool name. You need to understand the intent behind the query and select the best tool accordingly.

Allowed tools (use EXACT names only):
calculator
weather
summarize
greet
date

Rules:
- Output EXACTLY 2 lines
- No explanations
- Format:
  tool:<name>
  input:<value>
- If no input needed → input:none
- Never invent tool names

Mappings:
- math, numbers → calculator
- weather, temperature → weather
- summarize, summary → summarize
- hello, hi → greet
- date, today → date

Examples:

Query: 2+2
tool:calculator
input:2+2

Query: weather in mumbai
tool:weather
input:mumbai

Query: hello
tool:greet
input:none

Query: what is today's date
tool:date
input:none

Query: summarize: AI is powerful. It helps people.
tool:summarize
input:AI is powerful. It helps people.

Query: {query}
"""

    res = ollama.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}]
    )

    text = res["message"]["content"]

    tool, arg = None, ""

    for line in text.splitlines():
        line = line.strip().lower()
        if line.startswith("tool:"):
            tool = line.split("tool:")[1].strip()
        elif line.startswith("input:"):
            arg = line.split("input:")[1].strip()

    if arg == "none":
        arg = ""

    return tool, arg, text

In [11]:
def run_agent(query: str):
    tool, arg, raw = decide_tool_llm(query)

    if tool not in TOOLS:
        return {
            "input": query,
            "tool": tool,
            "output": "Invalid tool selected",
            "llm_raw": raw
        }

    result = TOOLS[tool](arg)

    return {
        "input": query,
        "tool": tool,
        "output": result
    }

In [12]:
while True:
    q = input("Enter command: ").strip()

    if q in {"exit", "quit"}:
        break

    res = run_agent(q)

    print("Tool  :", res["tool"])
    print("Output:", res["output"])
    print()

Tool  : greet
Output: Hello

Tool  : calculator
Output: 9+9 = 18

Tool  : weather
Output: Mumbai: 30°C, Humid & Hazy, 78%

Tool  : summarize
Output: ai is blah balh. hla.

Tool  : greet
Output: Hello

Tool  : date
Output: 2026-04-07

